# Suryodaya HR chatbot (Rust): RAG baseline

A self-contained notebook for the **Rust (evcxr)** kernel. It asks where your data is, builds a multilingual search
index over the HR-policy documents, runs a retrieval-augmented chatbot on a local Ollama model, and measures its
accuracy on the multi-hop test set.

**Needs:** the Ollama app running on this computer. Nothing else: the notebook downloads the models it needs
(about 5 GB the first time) and compiles its own code (a few minutes the first time, seconds afterwards).

**Run:** *Run All*. The *Data* cell opens the standard Windows file window (on top of the browser, starting in
Downloads): choose the data zip, or any file inside the folder you extracted it to.

In [ ]:
:opt 2
:dep anyhow = "1"
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep bincode = "1.3"
:dep ureq = { version = "2", default-features = false, features = ["json"] }
:dep zip = { version = "2", default-features = false, features = ["deflate"] }
:dep evcxr_input = "1"

## Settings

In [ ]:
const OLLAMA: &str = "http://localhost:11434";   // Ollama server
const EMBED_MODEL: &str = "bge-m3";               // multilingual embeddings (all 8 languages)
const CHAT_MODEL: &str = "gemma3:4b";             // main chatbot model
const SMALL_MODEL: &str = "gemma3:1b";            // small model (the one the fine-tuning step trains)
const EVERY: usize = 10;                          // accuracy runs use every 10th question (100 of 1,000)

## Library code

Tokenizer, chunker, BM25 + dense index, Ollama client, retrieval, the chatbot, scoring, and data loading. The cells are
collapsed; click a cell's left bar to read the code.

In [ ]:
pub mod text {
    /// Zero code points of the Indic digit blocks handled (Devanagari .. Kannada).
    const DIGIT_BLOCKS: [u32; 8] = [0x0966, 0x09E6, 0x0A66, 0x0AE6, 0x0B66, 0x0BE6, 0x0C66, 0x0CE6];

    /// Map any Indic digit to its ASCII digit.
    pub fn ascii_digit(c: char) -> char {
        let u = c as u32;
        for b in DIGIT_BLOCKS {
            if u >= b && u < b + 10 {
                return char::from_u32('0' as u32 + (u - b)).unwrap();
            }
        }
        c
    }

    /// Digits to ASCII, strip markdown emphasis (`**`, `__`, backticks), lowercase.
    pub fn norm(s: &str) -> String {
        let s: String = s.chars().map(ascii_digit).collect();
        s.replace("**", "").replace("__", "").replace('`', "").to_lowercase()
    }

    fn is_word_char(c: char) -> bool {
        let u = c as u32;
        c.is_alphanumeric()
            || c == '_'
            || (0x0900..=0x0963).contains(&u) // Devanagari letters + vowel signs (danda excluded)
            || (0x0966..=0x0D7F).contains(&u) // Bengali .. Malayalam blocks, incl. matras
            || c == '\u{200C}'
            || c == '\u{200D}'
    }

    /// Word tokens: maximal runs of word characters (Indic words kept whole).
    pub fn word_tokens(s: &str) -> Vec<String> {
        let n = norm(s);
        let mut out = Vec::new();
        let mut cur = String::new();
        for c in n.chars() {
            if is_word_char(c) {
                cur.push(c);
            } else if !cur.is_empty() {
                out.push(std::mem::take(&mut cur));
            }
        }
        if !cur.is_empty() {
            out.push(cur);
        }
        out
    }

    /// Character n-grams (n=4) over `#word#`, as in the Python index.
    pub fn char_grams(s: &str) -> Vec<String> {
        let n = 4;
        let mut out = Vec::new();
        for w in word_tokens(s) {
            let cs: Vec<char> = format!("#{w}#").chars().collect();
            if cs.len() <= n {
                out.push(cs.iter().collect());
            } else {
                for i in 0..=cs.len() - n {
                    out.push(cs[i..i + n].iter().collect());
                }
            }
        }
        out
    }

    /// Remove all whitespace after `norm` (used for verbatim evidence matching).
    pub fn squash(s: &str) -> String {
        norm(s).chars().filter(|c| !c.is_whitespace()).collect()
    }

    /// Split into sentences after । ॥ . ? ! followed by whitespace.
    pub fn sentences(s: &str) -> Vec<String> {
        let mut out = Vec::new();
        let mut cur = String::new();
        let chars: Vec<char> = s.chars().collect();
        let mut i = 0;
        while i < chars.len() {
            let c = chars[i];
            cur.push(c);
            if matches!(c, '।' | '॥' | '.' | '?' | '!') && i + 1 < chars.len() && chars[i + 1].is_whitespace() {
                let mut j = i + 1;
                while j < chars.len() && chars[j].is_whitespace() {
                    j += 1;
                }
                if j < chars.len() {
                    out.push(std::mem::take(&mut cur));
                    i = j;
                    continue;
                }
            }
            i += 1;
        }
        if !cur.is_empty() {
            out.push(cur);
        }
        out
    }

    /// All numbers (as f64) in a string, Indic digits normalised, thousands commas removed.
    pub fn numbers(s: &str) -> Vec<f64> {
        let t: String = s.chars().map(ascii_digit).filter(|&c| c != ',').collect();
        let cs: Vec<char> = t.chars().collect();
        let mut out = Vec::new();
        let mut i = 0;
        while i < cs.len() {
            if cs[i].is_ascii_digit() {
                let st = i;
                while i < cs.len() && cs[i].is_ascii_digit() {
                    i += 1;
                }
                if i + 1 < cs.len() && cs[i] == '.' && cs[i + 1].is_ascii_digit() {
                    i += 1;
                    while i < cs.len() && cs[i].is_ascii_digit() {
                        i += 1;
                    }
                }
                let tok: String = cs[st..i].iter().collect();
                if let Ok(v) = tok.parse::<f64>() {
                    out.push(v);
                }
            } else {
                i += 1;
            }
        }
        out
    }

    /// Detect the dominant Indic script of a string and return the language name.
    pub fn detect_language(s: &str) -> &'static str {
        let mut counts = [0usize; 8];
        for c in s.chars() {
            let u = c as u32;
            let idx = match u {
                0x0900..=0x097F => 0, // Devanagari: Hindi/Marathi (disambiguated below)
                0x0980..=0x09FF => 1,
                0x0A00..=0x0A7F => 2,
                0x0B00..=0x0B7F => 3,
                0x0B80..=0x0BFF => 4,
                0x0C00..=0x0C7F => 5,
                0x0C80..=0x0CFF => 6,
                _ => continue,
            };
            counts[idx] += 1;
        }
        let (best, n) = counts.iter().enumerate().max_by_key(|x| x.1).unwrap();
        if *n == 0 {
            return "English";
        }
        match best {
            0 => {
                // Marathi-specific markers
                if s.contains("आहे") || s.contains("च्या") || s.contains("ळ") { "Marathi" } else { "Hindi" }
            }
            1 => "Bengali",
            2 => "Punjabi",
            3 => "Odia",
            4 => "Tamil",
            5 => "Telugu",
            _ => "Kannada",
        }
    }
}

In [ ]:
pub mod chunk {
    use serde::{Deserialize, Serialize};
    use std::path::Path;

    pub const MAX_WORDS: usize = 220;

    #[derive(Clone, Debug, Serialize, Deserialize)]
    pub struct Chunk {
        pub id: String,
        pub language: String,
        pub document: String,
        pub doc_title: String,
        pub heading: String,
        pub text: String,
    }

    fn parse_heading(s: &str) -> Option<(usize, String)> {
        let hashes = s.chars().take_while(|&c| c == '#').count();
        if hashes == 0 || hashes > 6 {
            return None;
        }
        let rest = &s[hashes..];
        // Python: ^(#{1,6})\s+(.*)$  -> at least one whitespace after the hashes
        if !rest.starts_with(|c: char| c.is_whitespace()) {
            return None;
        }
        Some((hashes, rest.trim_start().to_string()))
    }

    pub fn chunk_doc(lang: &str, doc: &str, raw: &str) -> Vec<Chunk> {
        let mut title: Option<String> = None;
        let mut heading_path: Vec<String> = Vec::new();
        let mut blocks: Vec<(Vec<String>, String)> = Vec::new();
        let mut para: Vec<String> = Vec::new();
        let flush = |para: &mut Vec<String>, blocks: &mut Vec<(Vec<String>, String)>, hp: &Vec<String>| {
            if !para.is_empty() {
                blocks.push((hp.clone(), para.join(" ")));
                para.clear();
            }
        };
        for ln in raw.split('\n') {
            let s = ln.trim();
            if s.starts_with("<!--") {
                continue;
            }
            if let Some((lvl, txt)) = parse_heading(s) {
                flush(&mut para, &mut blocks, &heading_path);
                let txt = txt.replace("**", "").trim().to_string();
                if lvl == 2 && title.is_none() {
                    title = Some(txt);
                    heading_path.clear();
                    continue;
                }
                if lvl == 1 {
                    continue;
                }
                let keep = lvl.saturating_sub(3).min(heading_path.len());
                heading_path.truncate(keep);
                heading_path.push(txt);
                continue;
            }
            if s.is_empty() || s == "---" {
                flush(&mut para, &mut blocks, &heading_path);
                continue;
            }
            para.push(s.to_string());
        }
        flush(&mut para, &mut blocks, &heading_path);

        let mut out_raw: Vec<(String, String)> = Vec::new();
        let mut cur: Vec<String> = Vec::new();
        let mut cur_h: Option<Vec<String>> = None;
        let mut cur_n = 0usize;
        for (h, p) in blocks {
            let n = p.split_whitespace().count();
            if cur_h.as_ref() != Some(&h) || cur_n + n > MAX_WORDS {
                if !cur.is_empty() {
                    let hd = cur_h.as_ref().map(|v| v.join(" > ")).unwrap_or_default();
                    out_raw.push((hd, cur.join("\n")));
                }
                cur.clear();
                cur_n = 0;
                cur_h = Some(h);
            }
            cur.push(p);
            cur_n += n;
        }
        if !cur.is_empty() {
            let hd = cur_h.as_ref().map(|v| v.join(" > ")).unwrap_or_default();
            out_raw.push((hd, cur.join("\n")));
        }
        let prefix: String = lang.chars().take(2).collect::<String>().to_uppercase();
        let title = title.unwrap_or_else(|| doc.to_string());
        out_raw
            .into_iter()
            .enumerate()
            .map(|(i, (heading, text))| Chunk {
                id: format!("{prefix}:{doc}:{i:04}"),
                language: lang.to_string(),
                document: doc.to_string(),
                doc_title: title.clone(),
                heading,
                text,
            })
            .collect()
    }

    /// Chunk every `<corpus>/<Language>/NN_*.md`, in the same order as the Python builder.
    pub fn chunk_corpus(corpus: &Path) -> anyhow::Result<Vec<Chunk>> {
        let mut paths = Vec::new();
        for lang in std::fs::read_dir(corpus)? {
            let lang = lang?.path();
            if !lang.is_dir() {
                continue;
            }
            for f in std::fs::read_dir(&lang)? {
                let f = f?.path();
                let name = f.file_name().unwrap().to_string_lossy().to_string();
                let b = name.as_bytes();
                if name.ends_with(".md") && b.len() > 3 && b[0].is_ascii_digit() && b[1].is_ascii_digit() && b[2] == b'_' {
                    paths.push(f);
                }
            }
        }
        paths.sort();
        let mut chunks = Vec::new();
        for p in paths {
            let lang = p.parent().unwrap().file_name().unwrap().to_string_lossy().to_string();
            let doc = p.file_stem().unwrap().to_string_lossy().to_string();
            let raw = std::fs::read_to_string(&p)?;
            chunks.extend(chunk_doc(&lang, &doc, &raw));
        }
        Ok(chunks)
    }
}

In [ ]:
pub mod index {
    use crate::chunk::Chunk;
    use crate::text::{char_grams, word_tokens};
    use serde::{Deserialize, Serialize};
    use std::collections::HashMap;
    use std::path::Path;

    #[derive(Serialize, Deserialize)]
    pub struct Bm25 {
        k1: f32,
        b: f32,
        avg: f32,
        len: Vec<u32>,
        post: HashMap<String, Vec<(u32, u32)>>,
        idf: HashMap<String, f32>,
    }

    impl Bm25 {
        pub fn build(docs: &[Vec<String>]) -> Self {
            let n = docs.len();
            let len: Vec<u32> = docs.iter().map(|d| d.len() as u32).collect();
            let avg = len.iter().map(|&x| x as f32).sum::<f32>() / n.max(1) as f32;
            let mut post: HashMap<String, Vec<(u32, u32)>> = HashMap::new();
            for (i, toks) in docs.iter().enumerate() {
                let mut tf: HashMap<&str, u32> = HashMap::new();
                for t in toks {
                    *tf.entry(t.as_str()).or_default() += 1;
                }
                for (t, f) in tf {
                    post.entry(t.to_string()).or_default().push((i as u32, f));
                }
            }
            let idf = post
                .iter()
                .map(|(t, p)| {
                    let df = p.len() as f32;
                    (t.clone(), (1.0 + (n as f32 - df + 0.5) / (df + 0.5)).ln())
                })
                .collect();
            Bm25 { k1: 1.4, b: 0.75, avg, len, post, idf }
        }

        /// Scores for the distinct query tokens (query term frequency ignored, as in the Python index).
        pub fn score(&self, q: &[String], allowed: Option<&[bool]>) -> HashMap<u32, f32> {
            let mut sc: HashMap<u32, f32> = HashMap::new();
            let mut seen = std::collections::HashSet::new();
            for t in q {
                if !seen.insert(t) {
                    continue;
                }
                let (Some(p), Some(&idf)) = (self.post.get(t), self.idf.get(t)) else { continue };
                for &(i, f) in p {
                    if let Some(a) = allowed {
                        if !a[i as usize] {
                            continue;
                        }
                    }
                    let f = f as f32;
                    let dl = self.len[i as usize] as f32;
                    *sc.entry(i).or_default() += idf * f * (self.k1 + 1.0) / (f + self.k1 * (1.0 - self.b + self.b * dl / self.avg));
                }
            }
            sc
        }
    }

    #[derive(Serialize, Deserialize)]
    pub struct Store {
        pub chunks: Vec<Chunk>,
        pub word: Bm25,
        pub char: Bm25,
        /// Row-major, L2-normalised embeddings (chunks.len() x dim), if `embed` has been run.
        pub emb: Option<Vec<f32>>,
        pub dim: usize,
        pub emb_model: Option<String>,
        /// fingerprint of the corpus files the index was built from
        pub sig: String,
    }

    pub fn index_text(c: &Chunk) -> String {
        format!("{} {} {}", c.doc_title, c.heading, c.text)
    }

    impl Store {
        pub fn build(chunks: Vec<Chunk>) -> Self {
            let texts: Vec<String> = chunks.iter().map(index_text).collect();
            let word = Bm25::build(&texts.iter().map(|t| word_tokens(t)).collect::<Vec<_>>());
            let char = Bm25::build(&texts.iter().map(|t| char_grams(t)).collect::<Vec<_>>());
            Store { chunks, word, char, emb: None, dim: 0, emb_model: None, sig: String::new() }
        }
        pub fn save(&self, path: &Path) -> anyhow::Result<()> {
            if let Some(d) = path.parent() {
                std::fs::create_dir_all(d)?;
            }
            let f = std::io::BufWriter::new(std::fs::File::create(path)?);
            bincode::serialize_into(f, self)?;
            Ok(())
        }
        pub fn load(path: &Path) -> anyhow::Result<Self> {
            let f = std::io::BufReader::new(std::fs::File::open(path).map_err(|e| anyhow::anyhow!("{}: {e} (run `hr-rag index` first)", path.display()))?);
            Ok(bincode::deserialize_from(f)?)
        }
        pub fn pos(&self, id: &str) -> Option<usize> {
            self.chunks.iter().position(|c| c.id == id)
        }
    }
}

In [ ]:
pub mod ollama {
    use anyhow::{anyhow, Context, Result};
    use serde_json::{json, Value};
    use std::time::Duration;

    pub struct Ollama {
        base: String,
        agent: ureq::Agent,
    }

    #[derive(Clone, Debug)]
    pub struct Msg {
        pub role: &'static str,
        pub content: String,
    }

    impl Ollama {
        pub fn new(base: &str) -> Self {
            let agent = ureq::AgentBuilder::new()
                .timeout_connect(Duration::from_secs(10))
                .timeout_read(Duration::from_secs(900))
                .build();
            Ollama { base: base.trim_end_matches('/').to_string(), agent }
        }

        fn post(&self, path: &str, body: Value) -> Result<Value> {
            let url = format!("{}{}", self.base, path);
            let resp = self.agent.post(&url).send_json(body).map_err(|e| match e {
                ureq::Error::Status(code, r) => anyhow!("Ollama {url} returned {code}: {}", r.into_string().unwrap_or_default()),
                other => anyhow!("cannot reach Ollama at {url}: {other} (is `ollama serve` running?)"),
            })?;
            Ok(resp.into_json()?)
        }

        pub fn base(&self) -> &str {
            &self.base
        }

        /// Names of the locally installed models (e.g. "gemma3:4b", "bge-m3:latest").
        pub fn models(&self) -> Result<Vec<String>> {
            let url = format!("{}/api/tags", self.base);
            let v: Value = self
                .agent
                .get(&url)
                .call()
                .map_err(|e| anyhow!("cannot reach Ollama at {url}: {e} (start the Ollama app or run `ollama serve`)"))?
                .into_json()?;
            Ok(v["models"].as_array().map(|a| a.iter().filter_map(|m| m["name"].as_str().map(String::from)).collect()).unwrap_or_default())
        }

        /// Is `model` installed? ("bge-m3" matches "bge-m3:latest")
        pub fn has_model(&self, model: &str) -> Result<bool> {
            let want = if model.contains(':') { model.to_string() } else { format!("{model}:latest") };
            Ok(self.models()?.iter().any(|m| *m == want || m == model))
        }

        /// Download `model` unless it is installed already; prints progress to stderr.
        pub fn pull(&self, model: &str) -> Result<()> {
            if self.has_model(model)? {
                eprintln!("{model}: already installed");
                return Ok(());
            }
            let url = format!("{}/api/pull", self.base);
            let resp = self.agent.post(&url).send_json(json!({"model": model, "stream": true})).map_err(|e| anyhow!("pull {model} failed: {e}"))?;
            let mut last = String::new();
            for line in std::io::BufRead::lines(std::io::BufReader::new(resp.into_reader())) {
                let v: Value = serde_json::from_str(&line?)?;
                if let Some(e) = v["error"].as_str() {
                    return Err(anyhow!("pull {model}: {e}"));
                }
                let status = v["status"].as_str().unwrap_or("").to_string();
                let msg = match (v["completed"].as_u64(), v["total"].as_u64()) {
                    (Some(c), Some(t)) if t > 0 => format!("{status} {}%", c * 100 / t / 10 * 10),
                    _ => status,
                };
                if msg != last {
                    eprintln!("{model}: {msg}");
                    last = msg;
                }
            }
            if !self.has_model(model)? {
                return Err(anyhow!("pull {model} finished but the model is not listed"));
            }
            Ok(())
        }

        /// Embed a batch of texts; returns one vector per input.
        pub fn embed(&self, model: &str, inputs: &[String]) -> Result<Vec<Vec<f32>>> {
            let v = self.post("/api/embed", json!({"model": model, "input": inputs, "truncate": true}))?;
            let arr = v["embeddings"].as_array().context("no `embeddings` in /api/embed response")?;
            arr.iter()
                .map(|e| e.as_array().context("bad embedding").map(|x| x.iter().map(|f| f.as_f64().unwrap_or(0.0) as f32).collect()))
                .collect()
        }

        /// Chat completion. `schema` (a JSON schema) constrains the output to JSON when given.
        pub fn chat(&self, model: &str, msgs: &[Msg], schema: Option<&Value>, num_ctx: usize) -> Result<String> {
            let messages: Vec<Value> = msgs.iter().map(|m| json!({"role": m.role, "content": m.content})).collect();
            let mut body = json!({
                "model": model,
                "messages": messages,
                "stream": false,
                "keep_alive": "30m",
                "options": {"temperature": 0.0, "seed": 42, "num_ctx": num_ctx}
            });
            if let Some(s) = schema {
                body["format"] = s.clone();
            }
            let v = self.post("/api/chat", body)?;
            v["message"]["content"].as_str().map(|s| s.to_string()).context("no message.content in /api/chat response")
        }
    }
}

In [ ]:
pub mod retrieve {
    use crate::index::Store;
    use crate::ollama::Ollama;
    use crate::text::{char_grams, word_tokens};
    use anyhow::{bail, Result};
    use std::collections::HashMap;

    #[derive(Clone, Copy, Debug, PartialEq, Eq)]
    pub enum Mode {
        Bm25,
        Dense,
        Hybrid,
    }

    pub struct Retriever<'a> {
        pub store: &'a Store,
        pub ollama: Option<&'a Ollama>,
    }

    const RRF_K: f32 = 60.0;
    const DEPTH: usize = 200;

    fn ranked(sc: HashMap<u32, f32>) -> Vec<u32> {
        let mut v: Vec<(u32, f32)> = sc.into_iter().collect();
        v.sort_by(|a, b| b.1.partial_cmp(&a.1).unwrap().then(a.0.cmp(&b.0)));
        v.into_iter().take(DEPTH).map(|x| x.0).collect()
    }

    impl<'a> Retriever<'a> {
        fn allowed(&self, lang: Option<&str>, doc: Option<&str>) -> Option<Vec<bool>> {
            if lang.is_none() && doc.is_none() {
                return None;
            }
            Some(
                self.store
                    .chunks
                    .iter()
                    .map(|c| lang.map_or(true, |l| c.language.eq_ignore_ascii_case(l)) && doc.map_or(true, |d| c.document == d))
                    .collect(),
            )
        }

        fn dense_rank(&self, q: &str, allowed: Option<&[bool]>) -> Result<Vec<u32>> {
            let (Some(emb), Some(ol), Some(model)) = (&self.store.emb, self.ollama, &self.store.emb_model) else {
                bail!("dense retrieval needs embeddings: run `hr-rag embed` (and have Ollama running)")
            };
            let qv = ol.embed(model, &[q.to_string()])?.remove(0);
            let nq = qv.iter().map(|x| x * x).sum::<f32>().sqrt().max(1e-12);
            let d = self.store.dim;
            let mut sc = HashMap::new();
            for i in 0..self.store.chunks.len() {
                if let Some(a) = allowed {
                    if !a[i] {
                        continue;
                    }
                }
                let row = &emb[i * d..(i + 1) * d];
                let s: f32 = row.iter().zip(&qv).map(|(a, b)| a * b).sum::<f32>() / nq;
                sc.insert(i as u32, s);
            }
            Ok(ranked(sc))
        }

        /// Top-k chunk indices for `q`, optionally restricted to a language and/or document.
        pub fn search(&self, q: &str, lang: Option<&str>, doc: Option<&str>, k: usize, mode: Mode) -> Result<Vec<usize>> {
            let allowed = self.allowed(lang, doc);
            let a = allowed.as_deref();
            let mut lists: Vec<Vec<u32>> = Vec::new();
            if mode != Mode::Dense {
                lists.push(ranked(self.store.word.score(&word_tokens(q), a)));
                lists.push(ranked(self.store.char.score(&char_grams(q), a)));
            }
            if mode != Mode::Bm25 {
                lists.push(self.dense_rank(q, a)?);
            }
            let mut fused: HashMap<u32, f32> = HashMap::new();
            for l in &lists {
                for (r, &i) in l.iter().enumerate() {
                    *fused.entry(i).or_default() += 1.0 / (RRF_K + r as f32 + 1.0);
                }
            }
            let mut v: Vec<(u32, f32)> = fused.into_iter().collect();
            v.sort_by(|x, y| y.1.partial_cmp(&x.1).unwrap().then(x.0.cmp(&y.0)));
            Ok(v.into_iter().take(k).map(|x| x.0 as usize).collect())
        }
    }
}

In [ ]:
pub mod rag {
    use crate::index::Store;
    use crate::ollama::{Msg, Ollama};
    use crate::retrieve::{Mode, Retriever};
    use crate::text::detect_language;
    use anyhow::Result;
    use serde::{Deserialize, Serialize};
    use serde_json::{json, Value};
    use std::time::Instant;

    pub const LANGUAGES: [&str; 8] = ["Bengali", "Hindi", "Marathi", "Odia", "Punjabi", "Tamil", "Telugu", "Kannada"];

    #[derive(Clone, Copy, Debug, PartialEq, Eq)]
    pub enum Strategy {
        /// Retrieve once with the question, answer once.
        Simple,
        /// Plan per-fact queries (in each document's language), retrieve, answer; re-plan if facts are missing.
        Agentic,
    }

    #[derive(Clone, Debug)]
    pub struct Config {
        pub model: String,
        pub mode: Mode,
        pub strategy: Strategy,
        /// Chunks retrieved per query.
        pub k: usize,
        /// Maximum chunks placed in the answer prompt.
        pub max_ctx: usize,
        pub rounds: usize,
        pub num_ctx: usize,
    }

    impl Config {
        /// Defaults used for the reported runs: hybrid retrieval, agentic strategy, k 6, 12 passages, 2 rounds, 12k context.
        pub fn new(model: &str) -> Self {
            Config { model: model.to_string(), mode: Mode::Hybrid, strategy: Strategy::Agentic, k: 6, max_ctx: 12, rounds: 2, num_ctx: 12288 }
        }
        pub fn simple(mut self) -> Self {
            self.strategy = Strategy::Simple;
            self
        }
        pub fn agentic(mut self) -> Self {
            self.strategy = Strategy::Agentic;
            self
        }
        pub fn mode(mut self, m: Mode) -> Self {
            self.mode = m;
            self
        }
        /// Short run name such as "gemma3-4b_agentic_hybrid".
        pub fn name(&self) -> String {
            let m: String = self.model.chars().map(|c| if c.is_ascii_alphanumeric() || c == '.' || c == '-' { c } else { '-' }).collect();
            format!("{m}_{:?}_{:?}", self.strategy, self.mode).to_lowercase()
        }
    }

    #[derive(Clone, Debug, Default, Serialize, Deserialize)]
    pub struct Prediction {
        pub id: String,
        pub short_answer: String,
        pub answer: String,
        pub cited_chunks: Vec<String>,
        pub retrieved: Vec<String>,
        pub queries: Vec<String>,
        pub missing_info: String,
        pub llm_calls: usize,
        pub latency_ms: u128,
        #[serde(default)]
        pub error: Option<String>,
    }

    #[derive(Deserialize, Default)]
    struct AnswerOut {
        #[serde(default)]
        short_answer: String,
        #[serde(default)]
        answer: String,
        #[serde(default)]
        cited_chunks: Vec<String>,
        #[serde(default)]
        missing_info: String,
    }

    #[derive(Deserialize, Default)]
    struct PlanOut {
        #[serde(default)]
        queries: Vec<PlanQuery>,
    }
    #[derive(Deserialize, Default, Clone)]
    struct PlanQuery {
        #[serde(default)]
        query: String,
        #[serde(default)]
        language: String,
    }

    fn answer_schema() -> Value {
        json!({"type": "object",
               "properties": {"short_answer": {"type": "string"}, "answer": {"type": "string"},
                              "cited_chunks": {"type": "array", "items": {"type": "string"}},
                              "missing_info": {"type": "string"}},
               "required": ["short_answer", "answer", "cited_chunks", "missing_info"]})
    }
    fn plan_schema() -> Value {
        json!({"type": "object",
               "properties": {"queries": {"type": "array", "maxItems": 4, "items": {"type": "object",
                   "properties": {"query": {"type": "string"}, "language": {"type": "string", "enum": LANGUAGES}},
                   "required": ["query", "language"]}}},
               "required": ["queries"]})
    }

    pub const ANSWER_SYSTEM: &str = "You are the HR-policy assistant of Suryodaya Integrated Solutions Pvt. Ltd. \
    Answer ONLY from the numbered context passages; never use outside knowledge. A question may need facts from several \
    passages, possibly in different languages: find each fact, then combine them and do exactly the arithmetic or comparison \
    asked (keep working days and calendar days apart). Write in the language of the question. Fields: short_answer = the \
    minimal final answer (a number with its unit, yes/no with the decisive figure, a date, or a short phrase); answer = 2-4 \
    sentences with the figures you used and the reasoning; cited_chunks = ids of the passages you used; missing_info = a \
    short description of any needed fact that is NOT in the context (empty string if nothing is missing). Never invent a figure.";

    const PLAN_SYSTEM: &str = "You plan searches over the HR-policy documents of one company, written in 8 Indian languages. \
    Break the question into the separate facts it needs and write one short keyword-style search query per fact (at most 4). \
    Write each query in the language of the document most likely to contain that fact, using that document's own terms, and \
    give that language.";

    pub struct Rag<'a> {
        pub store: &'a Store,
        pub ollama: &'a Ollama,
        pub cfg: Config,
        doc_list: String,
    }

    fn parse<T: for<'de> Deserialize<'de> + Default>(s: &str) -> Option<T> {
        serde_json::from_str(s).ok().or_else(|| {
            let a = s.find('{')?;
            let b = s.rfind('}')?;
            serde_json::from_str(&s[a..=b]).ok()
        })
    }

    impl<'a> Rag<'a> {
        pub fn new(store: &'a Store, ollama: &'a Ollama, cfg: Config) -> Self {
            let mut seen = std::collections::BTreeSet::new();
            for c in &store.chunks {
                seen.insert(format!("{} | {}", c.language, c.doc_title));
            }
            let doc_list = seen.into_iter().collect::<Vec<_>>().join("\n");
            Rag { store, ollama, cfg, doc_list }
        }

        fn retriever(&self) -> Retriever<'_> {
            Retriever { store: self.store, ollama: Some(self.ollama) }
        }

        fn context(&self, ids: &[usize]) -> String {
            ids.iter()
                .map(|&i| {
                    let c = &self.store.chunks[i];
                    format!("[{}] ({} | {} | {})\n{}", c.id, c.language, c.doc_title, c.heading, c.text)
                })
                .collect::<Vec<_>>()
                .join("\n\n")
        }

        fn answer(&self, q: &str, ids: &[usize]) -> Result<AnswerOut> {
            let lang = detect_language(q);
            let msgs = [
                Msg { role: "system", content: ANSWER_SYSTEM.to_string() },
                Msg { role: "user", content: format!("Context passages:\n\n{}\n\nQuestion ({lang}): {q}", self.context(ids)) },
            ];
            let raw = self.ollama.chat(&self.cfg.model, &msgs, Some(&answer_schema()), self.cfg.num_ctx)?;
            Ok(parse::<AnswerOut>(&raw).unwrap_or(AnswerOut { short_answer: raw.chars().take(300).collect(), ..Default::default() }))
        }

        fn plan(&self, q: &str, missing: Option<&str>) -> Result<Vec<PlanQuery>> {
            let extra = missing.map(|m| format!("\n\nA first attempt could not find: {m}\nPlan new queries for the missing facts only.")).unwrap_or_default();
            let msgs = [
                Msg { role: "system", content: PLAN_SYSTEM.to_string() },
                Msg { role: "user", content: format!("Documents (language | title):\n{}\n\nQuestion: {q}{extra}", self.doc_list) },
            ];
            let raw = self.ollama.chat(&self.cfg.model, &msgs, Some(&plan_schema()), self.cfg.num_ctx)?;
            Ok(parse::<PlanOut>(&raw).unwrap_or_default().queries.into_iter().filter(|p| !p.query.trim().is_empty()).take(4).collect())
        }

        pub fn run(&self, id: &str, q: &str) -> Prediction {
            let t0 = Instant::now();
            let mut p = Prediction { id: id.to_string(), ..Default::default() };
            if let Err(e) = self.run_inner(q, &mut p) {
                p.error = Some(e.to_string());
            }
            p.latency_ms = t0.elapsed().as_millis();
            p
        }

        fn run_inner(&self, q: &str, p: &mut Prediction) -> Result<()> {
            let r = self.retriever();
            let c = &self.cfg;
            let mut ids: Vec<usize> = Vec::new();
            let push = |ids: &mut Vec<usize>, new: Vec<usize>| {
                for i in new {
                    if !ids.contains(&i) {
                        ids.push(i);
                    }
                }
            };
            match c.strategy {
                Strategy::Simple => {
                    push(&mut ids, r.search(q, None, None, c.max_ctx, c.mode)?);
                    let a = self.answer(q, &ids)?;
                    p.llm_calls = 1;
                    self.fill(p, a, &ids);
                }
                Strategy::Agentic => {
                    // the question itself is always one query; plan queries add one result list each
                    let mut lists: Vec<Vec<usize>> = vec![r.search(q, None, None, c.max_ctx.max(c.k), c.mode)?];
                    let mut missing: Option<String> = None;
                    for round in 0..c.rounds.max(1) {
                        let plan = self.plan(q, missing.as_deref())?;
                        p.llm_calls += 1;
                        for pq in &plan {
                            let lang = LANGUAGES.iter().find(|l| l.eq_ignore_ascii_case(pq.language.trim())).copied();
                            let tag = format!("{}: {}", lang.unwrap_or("any"), pq.query.trim());
                            if p.queries.contains(&tag) {
                                continue;
                            }
                            p.queries.push(tag);
                            lists.push(r.search(&pq.query, lang, None, c.k, c.mode)?);
                        }
                        // the question's own top half is always kept (so a weak plan can never do worse than simple
                        // retrieval on those slots); the rest is filled round-robin so every planned query contributes
                        ids.clear();
                        push(&mut ids, lists[0].iter().take(c.max_ctx / 2).copied().collect());
                        let depth = lists.iter().map(|l| l.len()).max().unwrap_or(0);
                        'fill: for d in 0..depth {
                            for l in &lists {
                                if let Some(&i) = l.get(d) {
                                    push(&mut ids, vec![i]);
                                    if ids.len() >= c.max_ctx {
                                        break 'fill;
                                    }
                                }
                            }
                        }
                        let a = self.answer(q, &ids)?;
                        p.llm_calls += 1;
                        let miss = a.missing_info.trim().to_string();
                        self.fill(p, a, &ids);
                        if miss.is_empty() || round + 1 == c.rounds.max(1) {
                            break;
                        }
                        missing = Some(miss);
                    }
                }
            }
            Ok(())
        }

        fn fill(&self, p: &mut Prediction, a: AnswerOut, ids: &[usize]) {
            p.short_answer = a.short_answer;
            p.answer = a.answer;
            p.missing_info = a.missing_info;
            let valid: std::collections::HashSet<&str> = ids.iter().map(|&i| self.store.chunks[i].id.as_str()).collect();
            p.cited_chunks = a.cited_chunks.into_iter().map(|s| s.trim().trim_matches(|c| c == '[' || c == ']').to_string()).filter(|s| valid.contains(s.as_str())).collect();
            p.retrieved = ids.iter().map(|&i| self.store.chunks[i].id.clone()).collect();
        }
    }
}

In [ ]:
pub mod eval {
    use crate::index::Store;
    use crate::rag::Prediction;
    use crate::text::{numbers, sentences, squash};
    use anyhow::Result;
    use serde::{Deserialize, Serialize};
    use std::collections::{BTreeMap, HashMap, HashSet};
    use std::path::Path;

    #[derive(Clone, Debug, Deserialize, Serialize)]
    pub struct Fact {
        pub language: String,
        pub document: String,
        pub subject: String,
        pub aspect: String,
    }

    #[derive(Clone, Debug, Deserialize, Serialize)]
    pub struct Item {
        pub id: String,
        pub language: String,
        pub hop_type: String,
        pub question: String,
        pub short_answer: String,
        pub answer: String,
        pub supporting_facts: Vec<Fact>,
    }

    pub fn load_testset(path: &Path) -> Result<Vec<Item>> {
        Ok(serde_json::from_str(&std::fs::read_to_string(path)?)?)
    }

    /// (language, document, subject, aspect) -> evidence text, from data/corpus/*/*.facts.json
    pub fn load_evidence(corpus: &Path) -> Result<HashMap<(String, String, String, String), String>> {
        #[derive(Deserialize)]
        struct F {
            subject: String,
            aspect: String,
            evidence: Option<String>,
        }
        let mut m = HashMap::new();
        for lang in std::fs::read_dir(corpus)? {
            let lang = lang?.path();
            if !lang.is_dir() {
                continue;
            }
            let l = lang.file_name().unwrap().to_string_lossy().to_string();
            for f in std::fs::read_dir(&lang)? {
                let f = f?.path();
                let name = f.file_name().unwrap().to_string_lossy().to_string();
                if let Some(doc) = name.strip_suffix(".facts.json") {
                    let v: Vec<F> = serde_json::from_str(&std::fs::read_to_string(&f)?)?;
                    for x in v {
                        m.insert((l.clone(), doc.to_string(), x.subject, x.aspect), x.evidence.unwrap_or_default());
                    }
                }
            }
        }
        Ok(m)
    }

    /// Gold chunk ids per supporting fact (same procedure as eval/retrieval_baseline.py).
    pub fn gold_chunks(store: &Store, items: &[Item], ev: &HashMap<(String, String, String, String), String>) -> HashMap<String, Vec<Vec<String>>> {
        let ctext: Vec<String> = store.chunks.iter().map(|c| squash(&c.text)).collect();
        let mut by_doc: HashMap<(String, String), Vec<usize>> = HashMap::new();
        for (i, c) in store.chunks.iter().enumerate() {
            by_doc.entry((c.language.clone(), c.document.clone())).or_default().push(i);
        }
        let mut out = HashMap::new();
        for it in items {
            let mut g = Vec::new();
            for f in &it.supporting_facts {
                let e = ev.get(&(f.language.clone(), f.document.clone(), f.subject.clone(), f.aspect.clone())).cloned().unwrap_or_default();
                let cand = by_doc.get(&(f.language.clone(), f.document.clone())).cloned().unwrap_or_default();
                let parts: Vec<String> = sentences(&e).into_iter().filter(|p| p.trim().chars().count() > 8).map(|p| squash(&p)).collect();
                let mut ids: Vec<String> = cand
                    .iter()
                    .filter(|&&i| parts.iter().any(|p| !p.is_empty() && ctext[i].contains(p.as_str())))
                    .map(|&i| store.chunks[i].id.clone())
                    .collect();
                ids.sort();
                if ids.is_empty() && !cand.is_empty() {
                    let evc: Vec<char> = squash(&e).chars().collect();
                    let last = if evc.len() > 40 { evc.len() - 40 } else { 1 };
                    let windows: Vec<String> = (0..last).step_by(20).map(|j| evc[j.min(evc.len())..(j + 40).min(evc.len())].iter().collect()).collect();
                    let score = |i: usize| windows.iter().filter(|w| !w.is_empty() && ctext[i].contains(w.as_str())).map(|w| w.chars().count()).max().unwrap_or(0);
                    let mut best = cand[0];
                    let mut bs = score(best);
                    for &i in &cand[1..] {
                        let s = score(i);
                        if s > bs {
                            best = i;
                            bs = s;
                        }
                    }
                    if bs > 0 {
                        ids = vec![store.chunks[best].id.clone()];
                    }
                }
                g.push(ids);
            }
            out.insert(it.id.clone(), g);
        }
        out
    }

    pub fn group_of(it: &Item) -> String {
        if it.hop_type == "cross_language" { "Cross-language".into() } else { it.language.clone() }
    }

    #[derive(Default, Clone, Serialize)]
    pub struct Row {
        pub n: usize,
        pub auto_n: usize,
        pub auto_ok: usize,
        pub facts: usize,
        pub facts_retrieved: usize,
        pub all_retrieved: usize,
        pub facts_cited: usize,
        pub all_cited: usize,
        pub errors: usize,
        pub judged: usize,
        pub judge_correct: usize,
        pub judge_partial: usize,
        pub llm_calls: usize,
        pub latency_ms: u128,
    }

    impl Row {
        pub fn pct(a: usize, b: usize) -> String {
            if b == 0 { "-".into() } else { format!("{:.1}", 100.0 * a as f64 / b as f64) }
        }
    }

    /// Is every number in the reference short answer present in the prediction?
    pub fn numeric_match(gold_short: &str, p: &Prediction) -> Option<bool> {
        let g = numbers(gold_short);
        if g.is_empty() {
            return None;
        }
        let mut pn = numbers(&p.short_answer);
        pn.extend(numbers(&p.answer));
        Some(g.iter().all(|x| pn.iter().any(|y| (x - y).abs() < 1e-9)))
    }

    pub fn score(
        items: &[Item],
        preds: &HashMap<String, Prediction>,
        gold: &HashMap<String, Vec<Vec<String>>>,
        judge: Option<&HashMap<String, String>>,
    ) -> BTreeMap<String, Row> {
        let mut rows: BTreeMap<String, Row> = BTreeMap::new();
        for it in items {
            let Some(p) = preds.get(&it.id) else { continue };
            let retrieved: HashSet<&str> = p.retrieved.iter().map(|s| s.as_str()).collect();
            let cited: HashSet<&str> = p.cited_chunks.iter().map(|s| s.as_str()).collect();
            let g: Vec<&Vec<String>> = gold.get(&it.id).map(|v| v.iter().filter(|x| !x.is_empty()).collect()).unwrap_or_default();
            let hit_r: Vec<bool> = g.iter().map(|ids| ids.iter().any(|i| retrieved.contains(i.as_str()))).collect();
            let hit_c: Vec<bool> = g.iter().map(|ids| ids.iter().any(|i| cited.contains(i.as_str()))).collect();
            let answered = !(p.short_answer.is_empty() && p.answer.is_empty());
            let auto = if answered { numeric_match(&it.short_answer, p) } else { None };
            let verdict = judge.and_then(|j| j.get(&it.id));
            for key in ["ALL".to_string(), format!("lang:{}", group_of(it)), format!("hop:{}", it.hop_type)] {
                let r = rows.entry(key).or_default();
                r.n += 1;
                if let Some(a) = auto {
                    r.auto_n += 1;
                    r.auto_ok += a as usize;
                }
                r.facts += hit_r.len();
                r.facts_retrieved += hit_r.iter().filter(|&&x| x).count();
                r.all_retrieved += hit_r.iter().all(|&x| x) as usize;
                r.facts_cited += hit_c.iter().filter(|&&x| x).count();
                r.all_cited += hit_c.iter().all(|&x| x) as usize;
                r.errors += p.error.is_some() as usize;
                r.llm_calls += p.llm_calls;
                r.latency_ms += p.latency_ms;
                if let Some(v) = verdict {
                    r.judged += 1;
                    r.judge_correct += (v == "correct") as usize;
                    r.judge_partial += (v == "partial") as usize;
                }
            }
        }
        rows
    }

    pub fn print_table(rows: &BTreeMap<String, Row>) -> String {
        let mut s = String::from(
            "| group | n | numeric match % | judge correct % | all facts retrieved % | fact recall (retrieved) % | all facts cited % | LLM calls/q | s/q | errors |\n|---|---|---|---|---|---|---|---|---|---|\n",
        );
        let mut keys: Vec<&String> = rows.keys().collect();
        keys.sort_by_key(|k| (k.as_str() != "ALL", !k.starts_with("lang:"), k.to_string()));
        for k in keys {
            let r = &rows[k];
            s.push_str(&format!(
                "| {} | {} | {} | {} | {} | {} | {} | {:.2} | {:.1} | {} |\n",
                k.trim_start_matches("lang:"),
                r.n,
                Row::pct(r.auto_ok, r.auto_n),
                Row::pct(r.judge_correct, r.judged),
                Row::pct(r.all_retrieved, r.n),
                Row::pct(r.facts_retrieved, r.facts),
                Row::pct(r.all_cited, r.n),
                r.llm_calls as f64 / r.n.max(1) as f64,
                r.latency_ms as f64 / 1000.0 / r.n.max(1) as f64,
                r.errors
            ));
        }
        s
    }

    pub fn read_predictions(path: &Path) -> Result<HashMap<String, Prediction>> {
        let mut m = HashMap::new();
        if path.exists() {
            for line in std::fs::read_to_string(path)?.lines() {
                if line.trim().is_empty() {
                    continue;
                }
                let p: Prediction = serde_json::from_str(line)?;
                m.insert(p.id.clone(), p);
            }
        }
        Ok(m)
    }
}

In [ ]:
pub mod session {
    use crate::eval::{self, Item, Row};
    use crate::index::{self, Store};
    use crate::ollama::Ollama;
    use crate::rag::{Config, Prediction, Rag};
    use crate::retrieve::{Mode, Retriever};
    use anyhow::{bail, Context, Result};
    use std::collections::{BTreeMap, HashMap};
    use std::io::Write;
    use std::path::{Path, PathBuf};

    fn esc(s: &str) -> String {
        s.replace('&', "&amp;").replace('<', "&lt;").replace('>', "&gt;")
    }

    fn show_html(h: &str) {
        println!("EVCXR_BEGIN_CONTENT text/html\n{h}\nEVCXR_END_CONTENT");
    }

    /// Accuracy report for one run (rows: ALL, one per question language, Cross-language, one per hop type).
    #[derive(Clone)]
    pub struct Report {
        pub name: String,
        pub rows: BTreeMap<String, Row>,
    }

    const COLS: [&str; 10] = ["group", "n", "numeric match %", "judge correct %", "all facts retrieved %", "fact recall %", "all facts cited %", "LLM calls/q", "s/q", "errors"];

    impl Report {
        fn keys(&self) -> Vec<&String> {
            let mut k: Vec<&String> = self.rows.keys().collect();
            k.sort_by_key(|k| (k.as_str() != "ALL", !k.starts_with("lang:"), k.to_string()));
            k
        }
        fn cells(r: &Row) -> Vec<String> {
            vec![
                r.n.to_string(),
                Row::pct(r.auto_ok, r.auto_n),
                Row::pct(r.judge_correct, r.judged),
                Row::pct(r.all_retrieved, r.n),
                Row::pct(r.facts_retrieved, r.facts),
                Row::pct(r.all_cited, r.n),
                format!("{:.2}", r.llm_calls as f64 / r.n.max(1) as f64),
                format!("{:.1}", r.latency_ms as f64 / 1000.0 / r.n.max(1) as f64),
                r.errors.to_string(),
            ]
        }
        /// Numeric-match accuracy (%) for a group ("ALL", "Hindi", "Cross-language", "hop:cross_document", ...).
        pub fn accuracy(&self, group: &str) -> Option<f64> {
            let r = self.rows.get(group).or_else(|| self.rows.get(&format!("lang:{group}")))?;
            (r.auto_n > 0).then(|| 100.0 * r.auto_ok as f64 / r.auto_n as f64)
        }
        pub fn markdown(&self) -> String {
            eval::print_table(&self.rows)
        }
        pub fn html(&self) -> String {
            let mut h = format!("<b>{}</b><table><tr>{}</tr>", esc(&self.name), COLS.iter().map(|c| format!("<th>{c}</th>")).collect::<String>());
            for k in self.keys() {
                let style = if k == "ALL" { " style=\"font-weight:bold\"" } else { "" };
                h += &format!("<tr{style}><td>{}</td>{}</tr>", esc(k.trim_start_matches("lang:")), Self::cells(&self.rows[k]).iter().map(|c| format!("<td>{c}</td>")).collect::<String>());
            }
            h + "</table>"
        }
        /// Rendered automatically by the evcxr Jupyter kernel when a cell ends with a Report.
        pub fn evcxr_display(&self) {
            show_html(&self.html());
        }
    }

    impl std::fmt::Display for Report {
        fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
            write!(f, "{}\n{}", self.name, self.markdown())
        }
    }

    /// Side-by-side summary of several runs (numeric match per language plus retrieval / citation figures).
    pub struct Comparison(pub String, pub String);
    impl Comparison {
        pub fn evcxr_display(&self) {
            show_html(&self.0);
        }
        pub fn markdown(&self) -> &str {
            &self.1
        }
    }
    impl std::fmt::Display for Comparison {
        fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
            write!(f, "{}", self.1)
        }
    }

    /// All per-group tables of several runs in one display (evcxr shows only one HTML block per cell).
    pub fn details(runs: &[&Report]) -> Comparison {
        Comparison(runs.iter().map(|r| r.html()).collect::<Vec<_>>().join("<br>"), runs.iter().map(|r| r.to_string()).collect::<Vec<_>>().join("\n"))
    }

    pub fn compare(runs: &[&Report]) -> Comparison {
        let langs = ["Bengali", "Hindi", "Kannada", "Marathi", "Odia", "Punjabi", "Tamil", "Telugu", "Cross-language"];
        let mut head = vec!["run".to_string(), "ALL".into()];
        head.extend(langs.iter().map(|s| s.to_string()));
        head.extend(["all facts retrieved %", "all facts cited %", "LLM calls/q", "s/q", "errors"].map(String::from));
        let mut rows = Vec::new();
        for r in runs {
            let a = r.rows.get("ALL").cloned().unwrap_or_default();
            let mut v = vec![r.name.clone(), r.accuracy("ALL").map_or("-".into(), |x| format!("{x:.1}"))];
            v.extend(langs.iter().map(|l| r.accuracy(l).map_or("-".into(), |x| format!("{x:.1}"))));
            v.push(Row::pct(a.all_retrieved, a.n));
            v.push(Row::pct(a.all_cited, a.n));
            v.push(format!("{:.2}", a.llm_calls as f64 / a.n.max(1) as f64));
            v.push(format!("{:.1}", a.latency_ms as f64 / 1000.0 / a.n.max(1) as f64));
            v.push(a.errors.to_string());
            rows.push(v);
        }
        let mut h = String::from("<b>numeric-match accuracy % by question language</b><table><tr>");
        h += &head.iter().map(|c| format!("<th>{}</th>", esc(c))).collect::<String>();
        h += "</tr>";
        for v in &rows {
            h += &format!("<tr>{}</tr>", v.iter().map(|c| format!("<td>{}</td>", esc(c))).collect::<String>());
        }
        h += "</table>";
        let mut md = format!("| {} |\n|{}\n", head.join(" | "), "---|".repeat(head.len()));
        for v in &rows {
            md += &format!("| {} |\n", v.join(" | "));
        }
        Comparison(h, md)
    }

    /// One chatbot answer with its sources; renders as HTML in Jupyter.
    pub struct Answer {
        pub question: String,
        pub prediction: Prediction,
        /// (language, document title, section heading) of each cited passage
        pub sources: Vec<(String, String, String)>,
        /// reference short answer, when the question comes from the test set
        pub reference: Option<String>,
    }

    impl Answer {
        pub fn evcxr_display(&self) {
            let p = &self.prediction;
            let mut h = format!("<div><i>{}</i><br><b>{}</b><p>{}</p>", esc(&self.question), esc(&p.short_answer), esc(&p.answer));
            if let Some(r) = &self.reference {
                h += &format!("<p><small>reference: {}</small></p>", esc(r));
            }
            if let Some(e) = &p.error {
                h += &format!("<p style=\"color:#b00\">error: {}</p>", esc(e));
            }
            if !self.sources.is_empty() {
                h += "<ul>";
                for (l, t, s) in &self.sources {
                    h += &format!("<li>{} | {} | {}</li>", esc(l), esc(t), esc(s));
                }
                h += "</ul>";
            }
            if !p.missing_info.trim().is_empty() {
                h += &format!("<p><small>not found: {}</small></p>", esc(&p.missing_info));
            }
            h += &format!("<small>{} LLM calls, {:.1} s; queries: {}</small></div>", p.llm_calls, p.latency_ms as f64 / 1000.0, esc(&p.queries.join(" | ")));
            show_html(&h);
        }
    }

    impl std::fmt::Display for Answer {
        fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
            let p = &self.prediction;
            writeln!(f, "{}\n\n{}\n\n{}", self.question, p.short_answer, p.answer)?;
            if let Some(r) = &self.reference {
                writeln!(f, "reference: {r}")?;
            }
            for (l, t, s) in &self.sources {
                writeln!(f, "  - {l} | {t} | {s}")?;
            }
            Ok(())
        }
    }

    /// Owns the index, the Ollama client, the test set and the gold evidence.
    pub struct Session {
        /// folder with one sub-folder per language holding NN_<topic>.md (and NN_<topic>.facts.json)
        pub corpus: PathBuf,
        /// the test set, if there is one (chat works without it; retrieval/eval need it)
        pub testset: Option<PathBuf>,
        /// where the index and the run files are written
        pub work: PathBuf,
        pub store_path: PathBuf,
        pub runs_dir: PathBuf,
        pub store: Store,
        pub ollama: Ollama,
        pub items: Vec<Item>,
        pub gold: HashMap<String, Vec<Vec<String>>>,
    }

    /// Remove the \\?\ prefix Windows puts on canonical paths, so paths print normally.
    pub fn plain(p: PathBuf) -> PathBuf {
        match p.to_str().and_then(|s| s.strip_prefix(r"\\?\")) {
            Some(rest) if !rest.starts_with("UNC") => PathBuf::from(rest),
            _ => p,
        }
    }

    /// Walk up from `start` to the folder that contains data/corpus and data/testset.
    pub fn find_repo(start: &Path) -> Result<PathBuf> {
        let start = if start.as_os_str().is_empty() { std::env::current_dir()? } else { start.to_path_buf() };
        let start = plain(std::fs::canonicalize(&start).with_context(|| format!("{} does not exist", start.display()))?);
        for d in start.ancestors() {
            if d.join("data/corpus").is_dir() && d.join("data/testset/suryodaya_qa_testset_1000.json").is_file() {
                return Ok(d.to_path_buf());
            }
        }
        bail!("no data/corpus and data/testset found in {} or any parent folder; pass the repository folder to Session::open", start.display())
    }

    /// Fingerprint of the corpus files (names and sizes), so a changed corpus triggers re-indexing.
    fn corpus_sig(corpus: &Path) -> Result<String> {
        let mut v: Vec<(String, u64)> = Vec::new();
        for lang in std::fs::read_dir(corpus)? {
            let lang = lang?.path();
            if !lang.is_dir() {
                continue;
            }
            for f in std::fs::read_dir(&lang)? {
                let f = f?;
                let name = f.file_name().to_string_lossy().to_string();
                if name.ends_with(".md") || name.ends_with(".facts.json") {
                    v.push((format!("{}/{name}", lang.file_name().unwrap().to_string_lossy()), f.metadata()?.len()));
                }
            }
        }
        v.sort();
        let mut h: u64 = 0xcbf29ce484222325;
        for (n, l) in &v {
            for b in n.bytes().chain(l.to_le_bytes()) {
                h ^= b as u64;
                h = h.wrapping_mul(0x100000001b3);
            }
        }
        Ok(format!("{}:{h:016x}", v.len()))
    }

    impl Session {
        /// Open the repository (`""` = current folder or any parent) with Ollama at localhost:11434.
        pub fn open(path: &str) -> Result<Session> {
            Self::open_with(path, "http://localhost:11434")
        }

        pub fn open_with(path: &str, ollama_url: &str) -> Result<Session> {
            let repo = find_repo(Path::new(path))?;
            Self::open_dirs(
                &repo.join("data").join("corpus"),
                Some(&repo.join("data").join("testset").join("suryodaya_qa_testset_1000.json")),
                &repo.join("rust"),
                ollama_url,
            )
        }

        /// Open explicit locations. The index is built on first use (about a second), reused afterwards,
        /// and rebuilt automatically if the corpus files change.
        pub fn open_dirs(corpus: &Path, testset: Option<&Path>, work: &Path, ollama_url: &str) -> Result<Session> {
            let store_path = work.join("store").join("index.bin");
            let runs_dir = work.join("runs");
            let sig = corpus_sig(corpus)?;
            let store = match Store::load(&store_path) {
                Ok(s) if s.sig == sig => s,
                _ => {
                    let mut s = Store::build(crate::chunk::chunk_corpus(corpus)?);
                    if s.chunks.is_empty() {
                        bail!("no NN_*.md documents found under {}", corpus.display());
                    }
                    s.sig = sig;
                    s.save(&store_path)?;
                    s
                }
            };
            let items = match testset {
                Some(t) => eval::load_testset(t)?,
                None => Vec::new(),
            };
            let gold = eval::gold_chunks(&store, &items, &eval::load_evidence(corpus)?);
            let s = Session {
                corpus: corpus.to_path_buf(),
                testset: testset.map(|t| t.to_path_buf()),
                work: work.to_path_buf(),
                store_path,
                runs_dir,
                store,
                ollama: Ollama::new(ollama_url),
                items,
                gold,
            };
            println!("{}", s.status());
            Ok(s)
        }

        pub fn status(&self) -> String {
            let docs: std::collections::BTreeSet<_> = self.store.chunks.iter().map(|c| (&c.language, &c.document)).collect();
            let langs: std::collections::BTreeSet<_> = self.store.chunks.iter().map(|c| &c.language).collect();
            format!(
                "corpus  {}\ntest set {}\nwork    {}\n{} documents in {} languages, {} chunks, {} test questions; embeddings: {}",
                self.corpus.display(),
                self.testset.as_ref().map_or("(none: chat only)".into(), |t| t.display().to_string()),
                self.work.display(),
                docs.len(),
                langs.len(),
                self.store.chunks.len(),
                self.items.len(),
                match (&self.store.emb_model, self.store.emb.is_some()) {
                    (Some(m), true) => format!("{m} (dim {})", self.store.dim),
                    _ => "none (BM25 only until embed() is run)".into(),
                }
            )
        }

        /// Re-chunk and re-index the corpus. Drops embeddings.
        pub fn rebuild_index(&mut self) -> Result<()> {
            let mut st = Store::build(crate::chunk::chunk_corpus(&self.corpus)?);
            st.sig = corpus_sig(&self.corpus)?;
            st.save(&self.store_path)?;
            self.store = st;
            self.gold = eval::gold_chunks(&self.store, &self.items, &eval::load_evidence(&self.corpus)?);
            println!("{}", self.status());
            Ok(())
        }

        fn need_testset(&self) -> Result<()> {
            if self.items.is_empty() {
                bail!("no test set loaded: retrieval and accuracy runs need one");
            }
            Ok(())
        }

        /// Check that Ollama is running and download any of `models` that are missing.
        pub fn ensure_models(&self, models: &[&str]) -> Result<()> {
            let have = self.ollama.models()?;
            println!("Ollama at {} has: {}", self.ollama.base(), if have.is_empty() { "(no models)".into() } else { have.join(", ") });
            for m in models {
                self.ollama.pull(m)?;
            }
            Ok(())
        }

        /// Embed all chunks with an Ollama embedding model and store the vectors in the index (skipped if already done).
        pub fn embed(&mut self, model: &str) -> Result<()> {
            if self.store.emb.is_some() && self.store.emb_model.as_deref() == Some(model) {
                println!("embeddings for {model} already in the index");
                return Ok(());
            }
            let texts: Vec<String> = self.store.chunks.iter().map(index::index_text).collect();
            let t0 = std::time::Instant::now();
            let mut all = Vec::with_capacity(texts.len() * 1024);
            let mut dim = 0;
            let batch = 16;
            for (b, part) in texts.chunks(batch).enumerate() {
                for v in self.ollama.embed(model, part)? {
                    let n = v.iter().map(|x| x * x).sum::<f32>().sqrt().max(1e-12);
                    dim = v.len();
                    all.extend(v.iter().map(|x| x / n));
                }
                let done = ((b + 1) * batch).min(texts.len());
                if b % 25 == 24 || done == texts.len() {
                    eprintln!("embedded {done}/{} ({:.0} s)", texts.len(), t0.elapsed().as_secs_f64());
                }
            }
            self.store.emb = Some(all);
            self.store.dim = dim;
            self.store.emb_model = Some(model.to_string());
            self.store.save(&self.store_path)?;
            println!("stored {} embeddings (dim {dim}) in {}", texts.len(), self.store_path.display());
            Ok(())
        }

        fn check_mode(&self, mode: Mode) -> Result<()> {
            if mode != Mode::Bm25 && self.store.emb.is_none() {
                bail!("mode {mode:?} needs embeddings: run session.embed(\"bge-m3\") first, or use Mode::Bm25");
            }
            Ok(())
        }

        /// Search the corpus; returns (chunk id, language, document title, heading, text).
        pub fn search(&self, query: &str, lang: Option<&str>, k: usize, mode: Mode) -> Result<Vec<(String, String, String, String, String)>> {
            self.check_mode(mode)?;
            let r = Retriever { store: &self.store, ollama: Some(&self.ollama) };
            Ok(r.search(query, lang, None, k, mode)?
                .into_iter()
                .map(|i| {
                    let c = &self.store.chunks[i];
                    (c.id.clone(), c.language.clone(), c.doc_title.clone(), c.heading.clone(), c.text.clone())
                })
                .collect())
        }

        /// Ask the chatbot one question (any of the 8 languages).
        pub fn ask(&self, question: &str, cfg: &Config) -> Result<Answer> {
            self.check_mode(cfg.mode)?;
            let p = Rag::new(&self.store, &self.ollama, cfg.clone()).run("chat", question);
            let sources = p
                .cited_chunks
                .iter()
                .filter_map(|id| self.store.pos(id))
                .map(|i| {
                    let c = &self.store.chunks[i];
                    (c.language.clone(), c.doc_title.clone(), c.heading.clone())
                })
                .collect();
            Ok(Answer { question: question.to_string(), prediction: p, sources, reference: None })
        }

        /// A test-set question by id, e.g. "HI-029".
        pub fn item(&self, id: &str) -> Result<&Item> {
            self.items.iter().find(|it| it.id == id).with_context(|| format!("no question {id}"))
        }

        /// Ask a test-set question and show the reference short answer next to the chatbot's.
        pub fn ask_item(&self, id: &str, cfg: &Config) -> Result<Answer> {
            let it = self.item(id)?;
            let mut a = self.ask(&it.question, cfg)?;
            a.reference = Some(it.short_answer.clone());
            Ok(a)
        }

        fn report(&self, name: &str, preds: &HashMap<String, Prediction>, judge: Option<&HashMap<String, String>>) -> Report {
            Report { name: name.to_string(), rows: eval::score(&self.items, preds, &self.gold, judge) }
        }

        /// Single-shot retrieval with the raw question (no LLM): evidence recall per group.
        pub fn retrieval(&self, mode: Mode, k: usize) -> Result<Report> {
            self.need_testset()?;
            self.check_mode(mode)?;
            let r = Retriever { store: &self.store, ollama: Some(&self.ollama) };
            let mut preds = HashMap::new();
            for (n, it) in self.items.iter().enumerate() {
                let ids = r.search(&it.question, None, None, k, mode)?;
                preds.insert(it.id.clone(), Prediction { id: it.id.clone(), retrieved: ids.iter().map(|&i| self.store.chunks[i].id.clone()).collect(), ..Default::default() });
                if mode != Mode::Bm25 && (n + 1) % 200 == 0 {
                    eprintln!("{}/{}", n + 1, self.items.len());
                }
            }
            Ok(self.report(&format!("retrieval only, {mode:?}, k={k}"), &preds, None))
        }

        /// The questions a run uses: every `every`-th question (10 -> 100 questions with the same language/hop mix).
        pub fn subset(&self, every: usize) -> Vec<&Item> {
            self.items.iter().step_by(every.max(1)).collect()
        }

        /// Run the test set (every `every`-th question) through the chatbot. Predictions are appended to
        /// rust/runs/<run name>.jsonl, so an interrupted run resumes where it stopped; the report is also written as .md.
        pub fn eval(&self, cfg: &Config, every: usize) -> Result<Report> {
            self.eval_named(cfg, every, &cfg.name())
        }

        pub fn eval_named(&self, cfg: &Config, every: usize, name: &str) -> Result<Report> {
            self.need_testset()?;
            self.check_mode(cfg.mode)?;
            std::fs::create_dir_all(&self.runs_dir)?;
            let out = self.runs_dir.join(format!("{name}.jsonl"));
            let done = eval::read_predictions(&out)?;
            let sel = self.subset(every);
            let todo: Vec<&&Item> = sel.iter().filter(|it| !done.contains_key(&it.id)).collect();
            eprintln!("{name}: {} questions, {} already done, {} to run", sel.len(), sel.len() - todo.len(), todo.len());
            let rag = Rag::new(&self.store, &self.ollama, cfg.clone());
            let mut f = std::fs::OpenOptions::new().create(true).append(true).open(&out)?;
            let t0 = std::time::Instant::now();
            let (mut ok, mut seen) = (0usize, 0usize);
            for (n, it) in todo.iter().enumerate() {
                let p = rag.run(&it.id, &it.question);
                writeln!(f, "{}", serde_json::to_string(&p)?)?;
                f.flush()?;
                if let Some(m) = eval::numeric_match(&it.short_answer, &p) {
                    seen += 1;
                    ok += m as usize;
                }
                if let Some(e) = &p.error {
                    eprintln!("{} ERROR {e}", it.id);
                }
                if (n + 1) % 10 == 0 || n + 1 == todo.len() {
                    let el = t0.elapsed().as_secs_f64();
                    eprintln!(
                        "[{}/{}] running numeric match {:.1}% | {:.1} s/q | about {:.0} min left",
                        n + 1,
                        todo.len(),
                        100.0 * ok as f64 / seen.max(1) as f64,
                        el / (n + 1) as f64,
                        el / (n + 1) as f64 * (todo.len() - n - 1) as f64 / 60.0
                    );
                }
            }
            let all = eval::read_predictions(&out)?;
            let ids: std::collections::HashSet<&str> = sel.iter().map(|it| it.id.as_str()).collect();
            let preds: HashMap<String, Prediction> = all.into_iter().filter(|(k, _)| ids.contains(k.as_str())).collect();
            let rep = self.report(name, &preds, None);
            std::fs::write(self.runs_dir.join(format!("{name}.md")), rep.markdown())?;
            Ok(rep)
        }

        /// Score an existing run (optionally with judge verdicts {id: {"label": ...}}).
        pub fn score(&self, name: &str, judge: Option<&str>) -> Result<Report> {
            let preds = eval::read_predictions(&self.runs_dir.join(format!("{name}.jsonl")))?;
            if preds.is_empty() {
                bail!("no predictions in {}", self.runs_dir.join(format!("{name}.jsonl")).display());
            }
            let j: Option<HashMap<String, String>> = match judge {
                Some(p) => {
                    let v: HashMap<String, serde_json::Value> = serde_json::from_str(&std::fs::read_to_string(p)?)?;
                    Some(v.into_iter().map(|(k, x)| (k, x["label"].as_str().unwrap_or("").to_string())).collect())
                }
                None => None,
            };
            Ok(self.report(name, &preds, j.as_ref()))
        }
    }
}

In [ ]:
pub mod data {
    use anyhow::{bail, Context, Result};
    use std::path::{Path, PathBuf};

    #[derive(Clone, Debug)]
    pub struct DataPaths {
        /// folder whose sub-folders (one per language) hold NN_<topic>.md documents
        pub corpus: PathBuf,
        /// test-set JSON, if one was found or chosen
        pub testset: Option<PathBuf>,
        /// folder for the index and run files (next to the corpus)
        pub work: PathBuf,
    }

    const SKIP: [&str; 6] = ["hr_rag_work", "target", "node_modules", "__MACOSX", "AppData", "$RECYCLE.BIN"];
    const MAX_DIRS: usize = 4000;

    fn home() -> Option<PathBuf> {
        std::env::var_os("USERPROFILE").or_else(|| std::env::var_os("HOME")).map(PathBuf::from)
    }

    /// Trim spaces and quotes (Windows "Copy as path" adds them), expand ~, and resolve a bare name such as
    /// "Downloads" against the current folder, the home folder and the usual user folders.
    pub fn clean(input: &str) -> PathBuf {
        let s = input.trim().trim_matches(|c| c == '"' || c == '\'').trim();
        if s.is_empty() {
            return PathBuf::new();
        }
        let p = match (s.strip_prefix('~'), home()) {
            (Some(rest), Some(h)) => h.join(rest.trim_start_matches(['/', '\\'])),
            _ => PathBuf::from(s),
        };
        if p.is_absolute() || p.exists() {
            return p;
        }
        if let Some(h) = home() {
            for base in [h.clone(), h.join("Downloads"), h.join("Desktop"), h.join("Documents"), h.join("OneDrive"), h.join("OneDrive").join("Desktop"), h.join("OneDrive").join("Documents")] {
                if base.join(&p).exists() {
                    return base.join(&p);
                }
            }
        }
        p
    }

    fn is_doc(name: &str) -> bool {
        let b = name.as_bytes();
        name.ends_with(".md") && b.len() > 5 && b[0].is_ascii_digit() && b[1].is_ascii_digit() && b[2] == b'_'
    }

    fn subdirs(d: &Path) -> Vec<PathBuf> {
        let mut v: Vec<PathBuf> = std::fs::read_dir(d)
            .map(|r| r.filter_map(|e| e.ok()).map(|e| e.path()).filter(|p| p.is_dir()).collect())
            .unwrap_or_default();
        v.retain(|p| {
            let n = p.file_name().map(|n| n.to_string_lossy().to_string()).unwrap_or_default();
            !n.starts_with('.') && !SKIP.contains(&n.as_str())
        });
        v.sort();
        v
    }

    /// Number of NN_*.md documents in the language sub-folders of `d`.
    fn corpus_docs(d: &Path) -> usize {
        subdirs(d)
            .iter()
            .map(|l| std::fs::read_dir(l).map(|r| r.filter_map(|e| e.ok()).filter(|e| is_doc(&e.file_name().to_string_lossy())).count()).unwrap_or(0))
            .sum()
    }

    /// Folders under `root` (breadth-first, at most `depth` levels and MAX_DIRS folders).
    fn walk(root: &Path, depth: usize) -> Vec<PathBuf> {
        let mut out = vec![root.to_path_buf()];
        let mut level = vec![root.to_path_buf()];
        for _ in 0..depth {
            let mut next = Vec::new();
            for d in &level {
                for s in subdirs(d) {
                    if out.len() >= MAX_DIRS {
                        return out;
                    }
                    out.push(s.clone());
                    next.push(s);
                }
            }
            level = next;
        }
        out
    }

    fn find_corpus(root: &Path) -> Option<PathBuf> {
        walk(root, 5)
            .into_iter()
            .map(|d| (corpus_docs(&d), d))
            .filter(|(n, _)| *n > 0)
            .max_by_key(|(n, d)| (*n, d.file_name().map_or(false, |x| x == "corpus"), std::cmp::Reverse(d.components().count())))
            .map(|(_, d)| d)
    }

    fn find_testset(root: &Path) -> Option<PathBuf> {
        let mut c: Vec<PathBuf> = walk(root, 4)
            .iter()
            .flat_map(|d| std::fs::read_dir(d).into_iter().flatten().filter_map(|e| e.ok()).map(|e| e.path()))
            .filter(|p| {
                let n = p.file_name().map(|n| n.to_string_lossy().to_lowercase()).unwrap_or_default();
                p.is_file() && n.ends_with(".json") && (n.contains("testset") || n.contains("test_set"))
            })
            .filter(|p| crate::eval::load_testset(p).map(|v| !v.is_empty()).unwrap_or(false))
            .collect();
        c.sort();
        c.into_iter().next()
    }

    /// Does this zip contain language folders with NN_*.md documents?
    fn zip_has_corpus(zip: &Path) -> bool {
        let Ok(f) = std::fs::File::open(zip) else { return false };
        let Ok(a) = zip::ZipArchive::new(std::io::BufReader::new(f)) else { return false };
        let found = a.file_names().any(|n| {
            let parts: Vec<&str> = n.split(['/', '\\']).collect();
            parts.len() >= 2 && is_doc(parts[parts.len() - 1])
        });
        found
    }

    /// Extract `zip` into a folder named after it (next to it), unless that was done already.
    pub fn unzip(zip: &Path) -> Result<PathBuf> {
        let stem = zip.file_stem().context("bad zip name")?.to_string_lossy().to_string();
        let dest = zip.parent().unwrap_or(Path::new(".")).join(&stem);
        if dest.is_dir() && find_corpus(&dest).is_some() {
            println!("using the copy already extracted in {}", dest.display());
            return Ok(dest);
        }
        if !zip_has_corpus(zip) {
            bail!("{} does not contain the HR documents (language folders with files like 01_recruitment.md)", zip.display());
        }
        let f = std::fs::File::open(zip).with_context(|| format!("cannot open {}", zip.display()))?;
        let mut a = zip::ZipArchive::new(std::io::BufReader::new(f)).with_context(|| format!("{} is not a valid zip file", zip.display()))?;
        // a zip whose entries all sit in one top folder named like the zip is extracted beside it, not one level deeper
        let prefix = format!("{stem}/");
        let self_contained = a.file_names().all(|n| n.replace('\\', "/").starts_with(&prefix));
        let target = if self_contained { zip.parent().unwrap_or(Path::new(".")).to_path_buf() } else { dest.clone() };
        println!("extracting {} -> {}", zip.display(), dest.display());
        a.extract(&target).with_context(|| format!("cannot extract into {}", target.display()))?;
        Ok(dest)
    }

    /// Zips directly inside `dir` that contain the documents (names with "suryodaya" first, then newest).
    fn zips_in(dir: &Path) -> Vec<PathBuf> {
        let mut v: Vec<(bool, std::time::SystemTime, PathBuf)> = std::fs::read_dir(dir)
            .into_iter()
            .flatten()
            .filter_map(|e| e.ok())
            .map(|e| e.path())
            .filter(|p| p.is_file() && p.extension().map_or(false, |e| e.eq_ignore_ascii_case("zip")))
            .filter(|p| zip_has_corpus(p))
            .map(|p| {
                let named = p.file_name().map_or(false, |n| n.to_string_lossy().to_lowercase().contains("suryodaya"));
                let t = p.metadata().and_then(|m| m.modified()).unwrap_or(std::time::UNIX_EPOCH);
                (named, t, p)
            })
            .collect();
        v.sort_by(|a, b| b.0.cmp(&a.0).then(b.1.cmp(&a.1)));
        v.into_iter().map(|x| x.2).collect()
    }

    fn finish(corpus: PathBuf) -> Result<DataPaths> {
        let base = corpus.parent().unwrap_or(&corpus).to_path_buf();
        // the test set is usually beside the corpus folder (data/testset) or one level further up
        let testset = find_testset(&base).or_else(|| base.parent().and_then(find_testset));
        let work = base.join("hr_rag_work");
        std::fs::create_dir_all(&work).with_context(|| format!("cannot write to {}", work.display()))?;
        Ok(DataPaths { corpus, testset, work })
    }

    /// Resolve a zip, a folder, or any file inside the data folder to the corpus, test set and work folder.
    pub fn locate_path(p: &Path) -> Result<DataPaths> {
        if p.as_os_str().is_empty() {
            bail!("no path given");
        }
        if !p.exists() {
            bail!("{} does not exist", p.display());
        }
        let p = crate::session::plain(std::fs::canonicalize(p)?);
        if p.is_file() && p.extension().map_or(false, |e| e.eq_ignore_ascii_case("zip")) {
            let root = unzip(&p)?;
            return finish(find_corpus(&root).context("the extracted zip has no documents")?);
        }
        let dir = if p.is_file() { p.parent().unwrap().to_path_buf() } else { p.clone() };
        // the folder itself (or something below it) holds the documents
        if let Some(c) = find_corpus(&dir) {
            return finish(c);
        }
        // a file in the folder that holds the downloaded zip (e.g. Downloads)
        if let Some(z) = zips_in(&dir).into_iter().next() {
            println!("using {}", z.display());
            let root = unzip(&z)?;
            return finish(find_corpus(&root).context("the extracted zip has no documents")?);
        }
        // a file deeper inside the extracted data (e.g. rust/README.md, or a document itself): go up through the
        // repository's own layout only, never into unrelated sibling folders
        for a in dir.ancestors().skip(1).take(3) {
            for c in [a.to_path_buf(), a.join("corpus"), a.join("data").join("corpus")] {
                if corpus_docs(&c) > 0 {
                    return finish(c);
                }
            }
        }
        bail!("no HR documents found at {} (expected the data zip, or a folder with language sub-folders holding files like 01_recruitment.md)", p.display())
    }

    pub fn locate(input: &str) -> Result<DataPaths> {
        locate_path(&clean(input))
    }

    // ---------------------------------------------------------------- native file window

    pub fn utf16_base64(s: &str) -> String {
        const T: &[u8; 64] = b"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/";
        let bytes: Vec<u8> = s.encode_utf16().flat_map(|u| u.to_le_bytes()).collect();
        let mut out = String::new();
        for ch in bytes.chunks(3) {
            let n = (ch[0] as u32) << 16 | (*ch.get(1).unwrap_or(&0) as u32) << 8 | *ch.get(2).unwrap_or(&0) as u32;
            for i in 0..4 {
                if i <= ch.len() {
                    out.push(T[(n >> (18 - 6 * i) & 63) as usize] as char);
                } else {
                    out.push('=');
                }
            }
        }
        out
    }

    /// PowerShell script for the standard Windows "Open" window, kept on top of other windows.
    pub fn dialog_script(title: &str, filter: &str, start: Option<&Path>) -> String {
        let q = |s: &str| s.replace('\'', "''");
        let start = match start {
            Some(p) => format!("$d.InitialDirectory = '{}'", q(&p.display().to_string())),
            None => "try { $d.InitialDirectory = (New-Object -ComObject Shell.Application).NameSpace('shell:Downloads').Self.Path } catch {}".into(),
        };
        format!(
            "Add-Type -AssemblyName System.Windows.Forms\n\
             [System.Windows.Forms.Application]::EnableVisualStyles()\n\
             [Console]::OutputEncoding = New-Object System.Text.UTF8Encoding $false\n\
             $d = New-Object System.Windows.Forms.OpenFileDialog\n\
             $d.Title = '{}'\n\
             $d.Filter = '{}'\n\
             $d.Multiselect = $false\n\
             {start}\n\
             $owner = New-Object System.Windows.Forms.Form -Property @{{TopMost = $true; ShowInTaskbar = $false}}\n\
             if ($d.ShowDialog($owner) -eq [System.Windows.Forms.DialogResult]::OK) {{ [Console]::Out.Write($d.FileName) }}\n\
             $owner.Dispose()\n",
            q(title),
            q(filter)
        )
    }

    /// Open the system file window. Ok(None) = cancelled; Err = no file window available here.
    pub fn pick_file(title: &str, filter: &str, start: Option<&Path>) -> Result<Option<PathBuf>> {
        let out = if cfg!(windows) {
            std::process::Command::new("powershell.exe")
                .args(["-NoProfile", "-NonInteractive", "-STA", "-ExecutionPolicy", "Bypass", "-EncodedCommand"])
                .arg(utf16_base64(&dialog_script(title, filter, start)))
                .output()
                .context("cannot start PowerShell to open the file window")?
        } else {
            let mut r = std::process::Command::new("zenity").args(["--file-selection", &format!("--title={title}")]).output();
            if r.is_err() {
                r = std::process::Command::new("kdialog").args(["--getopenfilename", ".", "--title", title]).output();
            }
            r.context("no file window available (zenity/kdialog not installed)")?
        };
        let s = String::from_utf8_lossy(&out.stdout).trim_start_matches('\u{feff}').trim().to_string();
        if s.is_empty() {
            if !out.status.success() && !out.stderr.is_empty() && cfg!(windows) {
                bail!("file window failed: {}", String::from_utf8_lossy(&out.stderr).trim());
            }
            return Ok(None);
        }
        Ok(Some(PathBuf::from(s)))
    }

    fn report(d: &DataPaths) {
        println!("corpus   {}", d.corpus.display());
        println!("test set {}", d.testset.as_ref().map_or("(none: chat only)".to_string(), |t| t.display().to_string()));
        println!("work     {}", d.work.display());
    }

    /// Choose the data in the system file window: the downloaded .zip, or any file inside the extracted folder.
    /// Falls back to typing a path (with `read`, e.g. evcxr_input::get_string) only if no file window can be shown.
    pub fn choose(read: impl Fn(&str) -> String) -> Result<DataPaths> {
        let title = "Choose the HR data: the downloaded .zip, or any file inside the extracted folder";
        let filter = "Data zip (*.zip)|*.zip|All files (*.*)|*.*";
        println!("A file window has opened: choose the data zip (or any file inside the extracted data folder).");
        let mut start: Option<PathBuf> = None;
        for _ in 0..3 {
            match pick_file(title, filter, start.as_deref()) {
                Ok(Some(p)) => match locate_path(&p) {
                    Ok(mut d) => {
                        if d.testset.is_none() {
                            println!("No test set next to the documents. Choose the test-set .json, or Cancel for chat only.");
                            if let Ok(Some(t)) = pick_file("Choose the test-set .json (Cancel = chat only)", "Test set (*.json)|*.json", d.corpus.parent()) {
                                match crate::eval::load_testset(&t) {
                                    Ok(v) if !v.is_empty() => d.testset = Some(t),
                                    _ => println!("{} is not a test set; continuing without one", t.display()),
                                }
                            }
                        }
                        report(&d);
                        return Ok(d);
                    }
                    Err(e) => {
                        println!("{e:#}\nChoose again.");
                        start = p.parent().map(|x| x.to_path_buf());
                    }
                },
                Ok(None) => bail!("nothing chosen; run this cell again to open the file window"),
                Err(e) => {
                    println!("{e:#}; type the path instead.");
                    return ask(read);
                }
            }
        }
        bail!("no usable data chosen; run this cell again")
    }

    /// Text fallback: ask for a path until a usable one is given; up to 5 tries.
    pub fn ask(read: impl Fn(&str) -> String) -> Result<DataPaths> {
        for _ in 0..5 {
            let input = read("Path to the data zip or folder (a bare name like Downloads is looked up in your user folders): ");
            match locate(&input) {
                Ok(mut d) => {
                    if d.testset.is_none() {
                        let t = clean(&read("No test set found. Path to the test-set .json (Enter = none, chat only): "));
                        if !t.as_os_str().is_empty() {
                            match crate::eval::load_testset(&t) {
                                Ok(v) if !v.is_empty() => d.testset = Some(t),
                                _ => println!("could not read a test set from {}; continuing without one", t.display()),
                            }
                        }
                    }
                    report(&d);
                    return Ok(d);
                }
                Err(e) => println!("{e:#}. Try again."),
            }
        }
        bail!("no usable data location given")
    }
}

In [ ]:
use anyhow::Result;
use data::DataPaths;
use rag::Config;
use retrieve::Mode;
use session::{compare, details, Report, Session};
println!("library ready");

## 1. Data

A file window opens: choose the data **zip** (it is extracted next to itself, once), or **any file inside** the extracted
data folder. If the test set is not next to the documents, a second window asks for it (Cancel = chat only). The index
and all results are written to a `hr_rag_work` folder beside the documents.

In [ ]:
let paths: DataPaths = data::choose(evcxr_input::get_string)?;

In [ ]:
let mut s: Session = Session::open_dirs(&paths.corpus, paths.testset.as_deref(), &paths.work, OLLAMA)?;

## 2. Retrieval with keywords only (no model needed)

Single-shot BM25 (word + character 4-gram) with the raw question as the query. *all facts retrieved* = every gold
supporting fact is in the top 10; *fact recall* = share of gold facts in the top 10.

In [ ]:
s.retrieval(Mode::Bm25, 10)?

## 3. Models

Checks Ollama and downloads whatever is missing (bge-m3 about 1.2 GB, gemma3:4b about 3.3 GB, gemma3:1b about 0.8 GB).

In [ ]:
s.ensure_models(&[EMBED_MODEL, CHAT_MODEL, SMALL_MODEL])?;

## 4. Dense embeddings and hybrid retrieval

Embeds every chunk once (a few minutes on the GPU) and saves the vectors with the index; re-running skips it.

In [ ]:
s.embed(EMBED_MODEL)?;

In [ ]:
s.retrieval(Mode::Hybrid, 10)?

## 5. Chatbot

`Config::new(model)`: agentic strategy (plans one search per needed fact, in the language of the document that holds
it), hybrid retrieval, up to 12 passages. `.simple()` = one search and one answer. Three test questions first (the
reference answer is shown under the bot's), then a chat box: type questions in any of the 8 languages, empty line to stop.

In [ ]:
let bot: Config = Config::new(CHAT_MODEL);

In [ ]:
s.ask_item("HI-029", &bot)?

In [ ]:
s.ask_item("PA-069", &bot)?

In [ ]:
s.ask_item("XL-REC-001", &bot)?

In [ ]:
loop {
    let q: String = evcxr_input::get_string("Your question (empty = stop): ");
    if q.trim().is_empty() { break; }
    println!("{}\n", s.ask(q.trim(), &bot)?);
    std::io::Write::flush(&mut std::io::stdout())?;
    std::thread::sleep(std::time::Duration::from_millis(400)); // let the answer reach the page before the next box
};

## 6. Accuracy

Every `EVERY`-th question (100 questions, same language and hop-type mix as the full set). Each run saves its answers
in `hr_rag_work/runs/<run>.jsonl` and resumes where it stopped if interrupted: just run the cell again. *numeric match* =
every number in the reference short answer appears in the bot's answer.

In [ ]:
let grid: Vec<Config> = vec![
    Config::new(CHAT_MODEL).simple().mode(Mode::Bm25),
    Config::new(CHAT_MODEL).simple(),
    Config::new(CHAT_MODEL),
    Config::new(SMALL_MODEL),
];
let mut reports: Vec<Report> = Vec::new();

In [ ]:
reports.clear();
for c in &grid {
    reports.push(s.eval(c, EVERY)?);
};

In [ ]:
compare(&reports.iter().collect::<Vec<_>>())

Per-language and per-hop-type detail of each run (also saved as `hr_rag_work/runs/<run>.md`):

In [ ]:
details(&reports.iter().collect::<Vec<_>>())

## 7. Full test set (optional, several hours)

Set `FULL` to `true` to run all 1,000 questions with the agentic bot on both models. Resumable.

In [ ]:
const FULL: bool = false;
if FULL {
    let full: Vec<Report> = vec![
        s.eval_named(&Config::new(CHAT_MODEL), 1, "full_agentic_main")?,
        s.eval_named(&Config::new(SMALL_MODEL), 1, "full_agentic_small")?,
    ];
    compare(&full.iter().collect::<Vec<_>>()).evcxr_display();
};